In [28]:
import pandas as pd
import glob
import numpy as np
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm
import torch
import sys
import os
from pprint import pprint

# Add the parent directory to sys.path so src can be imported
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import src.utils as utils

In [5]:
# Leer archivo parquet
chunks_df = pd.read_parquet(r"..\data\processed\chunks.parquet")

# Ver columnas
print("Columnas disponibles:", chunks_df.columns)

Columnas disponibles: Index(['id_doc', 'autor_doc', 'fecha_doc', 'diario_doc', 'titulo_doc',
       'chunk_id', 'texto_chunk'],
      dtype='object')


In [6]:
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

In [8]:
# PARTE 1: Cargar términos y crear embeddings por subcategoría

def cargar_subcategorias(path_dir):
    """
    Lee los archivos .txt en el directorio, 
    genera embeddings de cada término con sentence-transformers,
    y devuelve el embedding promedio de cada subcategoría.
    """
    subcat_embeddings = {}
    for file_path in glob.glob(os.path.join(path_dir, "*.txt")):
        subcat = os.path.splitext(os.path.basename(file_path))[0]
        with open(file_path, "r", encoding="utf-8") as f:
            terms = [line.strip() for line in f if line.strip()]
        
        term_embeddings = model.encode(terms, convert_to_tensor=True, show_progress_bar=False)
        avg_embedding = term_embeddings.mean(dim=0)  # promedio de embeddings
        subcat_embeddings[subcat] = avg_embedding
    
    return subcat_embeddings

# PARTE 2: Calcular embeddings chunks

def obtener_embeddings_chunks(
    chunks_df,
    model,
    batch_size=64,
    save_path="../data/processed/chunk_embeddings.npy",
    RELOAD=False
):
    """
    Calcula o carga embeddings de los chunks según el valor de RELOAD.
    
    Args:
        chunks_df (pd.DataFrame): DataFrame con columna 'texto_chunk'
        model: modelo de sentence-transformers
        batch_size (int): tamaño del lote para encoding
        save_path (str): ruta donde guardar/cargar embeddings
        RELOAD (bool): si True, recalcula embeddings aunque exista archivo
    
    Returns:
        np.ndarray con embeddings de los chunks
    """
    
    if not RELOAD and os.path.exists(save_path):
        print(f"Embeddings encontrados en {save_path}, cargando...")
        embeddings = np.load(save_path)
        print(f"Embeddings cargados con forma: {embeddings.shape}")
        return embeddings
    
    print("Calculando embeddings desde cero...")
    textos = chunks_df["texto_chunk"].tolist()
    chunk_embeddings = model.encode(
        textos,
        batch_size=batch_size,
        convert_to_tensor=True,
        show_progress_bar=True
    )

    embeddings_np = chunk_embeddings.cpu().numpy()

    # Crear carpeta si no existe
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    # Guardar embeddings
    np.save(save_path, embeddings_np)
    print(f"Embeddings calculados y guardados en {save_path}")

    return embeddings_np

# PARTE 3: Calcular similitudes

def calcular_similitudes_chunks(chunks_df, chunk_embeddings, subcat_embeddings):
    """
    Calcula similitudes coseno entre embeddings de chunks y de subcategorías.
    
    Args:
        chunks_df (pd.DataFrame): DataFrame original con 'chunk_id', 'id_doc', 'texto_chunk'
        chunk_embeddings (np.ndarray o tensor): embeddings precomputados
        subcat_embeddings (dict): {subcat: tensor de embedding promedio}
    
    Returns:
        pd.DataFrame con similitudes por subcategoría
    """
    # Convertir a tensores
    chunk_embeddings = torch.tensor(chunk_embeddings)
    subcats = list(subcat_embeddings.keys())
    subcat_matrix = torch.stack(list(subcat_embeddings.values()))

    # Calcular similitudes en bloque
    sim_matrix = util.cos_sim(chunk_embeddings, subcat_matrix).cpu().numpy()

    # Armar DataFrame de resultados
    sim_df = pd.DataFrame(sim_matrix, columns=subcats)
    sim_df.insert(0, "chunk_id", chunks_df["chunk_id"].values)
    sim_df.insert(1, "id_doc", chunks_df["id_doc"].values)
    sim_df.insert(2, "texto_chunk", chunks_df["texto_chunk"].values)

    return sim_df


# PARTE 4: Aplicar umbral y asignar categorías

def asignar_categorias(df, umbral=0.30):
    """
    Añade columnas con las categorías detectadas por chunk
    y sus scores.
    """
    categorias_detectadas = []
    for _, fila in df.iterrows():
        cats = []
        for col in df.columns:
            if col not in ["chunk_id", "id_doc", "texto_chunk"]:
                if fila[col] >= umbral:
                    cats.append((col, fila[col]))
        categorias_detectadas.append(cats if cats else [("ninguna", 0)])
    
    df["categorias_detectadas"] = categorias_detectadas
    return df

In [9]:
# 1. Cargar embeddings de subcategorías
dir_tesauro = r"..\data\external\terminos\Tesauro_Literatura"
subcat_embeddings = cargar_subcategorias(dir_tesauro)


In [10]:
# 2. Calcular embeddings de chunks y guardarlos
chunk_embeddings = obtener_embeddings_chunks(
    chunks_df,
    model,
    batch_size=64,
    RELOAD=False   # cambiar a True si quieres forzar el recálculo
)

Embeddings encontrados en ../data/processed/chunk_embeddings.npy, cargando...
Embeddings cargados con forma: (35319, 384)


In [11]:
# 3. Calcular similitudes con subcategorías
sim_df = calcular_similitudes_chunks(
    chunks_df,
    chunk_embeddings,
    subcat_embeddings
)

In [12]:
# Definir ruta de salida
results_dir = r"..\data\results"
os.makedirs(results_dir, exist_ok=True)  # crear carpeta si no existe

output_path = os.path.join(results_dir, "similitudes_chunks_literatura.xlsx")

# Guardar el DataFrame en Excel
sim_df.to_excel(output_path, index=False, engine="openpyxl")

print(f"Resultados guardados en {output_path}")

Resultados guardados en ..\data\results\similitudes_chunks_literatura.xlsx


In [14]:
# 4. Asignar categorías con umbral
umbral_elegido = 0.4
resultado = asignar_categorias(sim_df, umbral=umbral_elegido)
resultado.head()

,chunk_id,id_doc,texto_chunk,Literatura,categorias_detectadas
0,0,1,"La Coalición Colombia Partido Alianza Verde, P...",0.230304,"[(ninguna, 0)]"
1,1,1,posible costo electoral que pocos asumen. Mani...,0.057526,"[(ninguna, 0)]"
2,0,2,Las interpretaciones de la historia sirven com...,0.361721,"[(ninguna, 0)]"
3,1,2,"y culturales, y del poder político: la hoy Mac...",0.171246,"[(ninguna, 0)]"
4,2,2,"Olimpo. En gracia de discusión, y solamente co...",0.248995,"[(ninguna, 0)]"


In [22]:
resultado_literatura = resultado[resultado['categorias_detectadas'].apply(lambda x: x[0][0] != 'ninguna')]

In [24]:
# Contar cuantos documentos tienen al menos una categoría asignada
docs_con_categoria = resultado[resultado['categorias_detectadas'].apply(lambda x: x[0][0] != 'ninguna')]['id_doc'].nunique()
docs_con_categoria

1771

In [25]:
resultado_literatura.head(10)

,chunk_id,id_doc,texto_chunk,Literatura,categorias_detectadas
16,1,6,le reconoce a las ideas de esas mismas persona...,0.441955,"[(Literatura, 0.4419548213481903)]"
21,0,9,El año viejo se llevó a un gran poeta quien a ...,0.581984,"[(Literatura, 0.581984281539917)]"
22,1,9,"cultivar ese género, a jugar con las palabras,...",0.496086,"[(Literatura, 0.49608585238456726)]"
28,0,12,Amordazar es taparle la boca a alguien con cua...,0.485784,"[(Literatura, 0.4857836663722992)]"
30,2,12,respete el derecho a defenderme de semejante c...,0.459778,"[(Literatura, 0.45977818965911865)]"
75,0,30,Recién durante las últimas horas del finiquita...,0.584160,"[(Literatura, 0.5841598510742188)]"
82,0,33,El que diga Uribe; cualquier cosa que escriba ...,0.634461,"[(Literatura, 0.6344612836837769)]"
83,1,33,"como dice el autor, de un relato histórico obs...",0.474277,"[(Literatura, 0.4742766320705414)]"
89,0,36,"Según el discurso victimista, la violencia psi...",0.531635,"[(Literatura, 0.5316352248191833)]"
96,0,39,Circula en las redes sociales la carta de un p...,0.425364,"[(Literatura, 0.42536449432373047)]"


In [29]:
# Juega con el n para ver otros fragmentos
n = 0
pprint(resultado_literatura.iloc[0]['texto_chunk'])

('le reconoce a las ideas de esas mismas personas en la formación nacional. '
 'Una y otra orientación deben ser la base para el avance y el mejoramiento de '
 'las condiciones de vida de la población afrodescendiente. La muestra '
 'contiene objetos sobre las características y horrores de la trata atlántica, '
 'la resistencia contra la esclavización, las formas de producción que las '
 'personas esclavizadas idearon en el litoral Pacífico, las relaciones que han '
 'establecido con plantas y animales de riberas y selvas húmedas, sus '
 'instrumentos musicales, cantos y liturgias para despedir a los muertos o '
 'para expiar dolores como el de la masacre de Bojayá. Otro valioso tributo '
 'consiste en la ilustración del pensamiento de las personas negras que ha '
 'liderado grandes transformaciones, como el movimiento de los derechos '
 'civiles, el derribamiento del apartheid o el movimiento social '
 'afrocolombiano. Entre esas primeras ideas está la de no cuentes los días, '
 'has 